## Create an agent


In [2]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

def get_weather(city: str) -> str:
    """Get weather for a give city."""
    return (f"It's always sunny in {city}")

model = ChatOpenAI(model="gpt-4o")

# Static prompt
agent = create_react_agent(
    model=model,
    tools=[get_weather],
    prompt="You are a helpful assistant"
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='36d81327-7279-443b-a4c6-cb36b5a5d8e2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_G26aJr8iEuppq5Pvipvnzl4P', 'function': {'arguments': '{"city":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 56, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1827dd0c55', 'id': 'chatcmpl-CCiUhcWaR6AoHL2bwp9lWlaeLsZ8f', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--0d35ebf6-ed9f-47cc-9ce5-0b3f54ed42c7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id':

## Add a dynamic prompt

In [3]:
from langchain_core.messages import AnyMessage
from langchain_core.runnables import RunnableConfig
from langgraph.prebuilt.chat_agent_executor import AgentState

def prompt(state:AgentState, config: RunnableConfig) -> list[AnyMessage]:
    user_name = config['configurable'].get("user_name")
    system_msg = f"You are a helpful assistant. Address the user as {user_name}."
    return [{"role": "system", "content": system_msg}] + state["messages"]

agent = create_react_agent(
    model=model,
    tools=[get_weather],
    prompt=prompt
)

agent.invoke(
     {"messages": [{"role": "user", "content": "what is the weather in sf"}]},
     config={"configurable": {"user_name": "John Smith"}}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='c8d849e9-318e-4ca3-9083-f865c48901ae'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5S90H8tkL98L5kCVhfFA0fjF', 'function': {'arguments': '{"city":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 63, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f33640a400', 'id': 'chatcmpl-CCiUmOjxEc5tlfSrfF7d30Y0AnTZb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--e5847dc7-6a0d-4bd9-b5ec-53243c6c743d-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id':

## Add memory

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

checkpoint = InMemorySaver()

agent = create_react_agent(
    model=model,
    tools=[get_weather],
    checkpointer=checkpoint
)

# Run the agent
config = {"configurable": {"thread_id": "1"}}
sf_response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in sf"}]},
    config
)

ny_response = agent.invoke(
    {"messages": [{"role": "user", "content": "What about new york?"}]},
    config
)

print(sf_response)
print()
print(ny_response)

{'messages': [HumanMessage(content='What is the weather in sf', additional_kwargs={}, response_metadata={}, id='2f98a46b-0d10-431d-b69f-ad0713e4b6b5'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_T0p6t5lqbmUuXW1pHfwSNbcQ', 'function': {'arguments': '{"city":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 50, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cbf1785567', 'id': 'chatcmpl-CCif4jETbhxs4Ydzkef6BpwqyY9BU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--f8e72eca-03f8-4717-9fd5-8df6da18584f-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '

## Configure structured output

In [8]:
from pydantic import BaseModel
from langgraph.prebuilt import create_react_agent

class WeatherResponse(BaseModel):
    condition: str
    
agent = create_react_agent(
    model=model,
    tools=[get_weather],
    response_format=WeatherResponse
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

response["structured_response"]

WeatherResponse(condition='Although the city is known for its microclimates, offering different weather patterns across its neighborhood, you can generally expect a cool and mild environment. Expect some fog in the morning and evening, with the possibility of quick shifts between sun and cloud cover throughout the day. \n  \n  - **Temperature:** Typically ranges between 55°F to 65°F (13°C to 18°C).\n  - **Chance of Rain:** Minimal, though occasional light showers occur in the wetter months.\n  - **Wind:** Often breezy, particularly near the coast.\n  - **Dress Accordingly:** Layers are essential, with a light jacket or a sweatshirt recommended even in the summer.\n  \n  Please make sure to check a reliable weather app or website for real-time updates and accuracy as the conditions can change swiftly in this vibrant city.')